# Digital Twin Data, Streaming, and Aggregation


## Photodetector Digital Twin Infrastructure

This notebook demonstrates real-time streaming acquisition from multiple transport protocols (ZMQ and MQTT), microservice socket integration, multi-stream data aggregation, physical state conversion, and verifying time-series records retrieved from the InfluxDB store.

--- 
## 1. Digital Twin State Definition

The digital twin state represents the physical condition and operational health of the photodetector asset. It is synthesized continuously by aggregating primitive streams (temperature_c, optical_power_w) across multiple transport protocols and evaluating them through optoelectronic physics models.

### State Schema Table

| State Field | Data Type | Units / Format | Primary Source | System Function / Role |
| :--- | :--- | :--- | :--- | :--- |
| `device_id` | `string` | Asset ID (`pd_sensor_01`) | Ingestion Stream (ZMQ / MQTT) | Primary key for physical asset telemetry |
| `temperature_c` | `float` | °C (24.5 - 25.5 °C nominal) | Ingestion Stream (ZMQ / MQTT) | Thermal sensor reading driving exponential dark current drift |
| `bias_voltage_v` | `float` | Volts (5.0 V baseline) | Ingestion Default | Ingested baseline bias voltage setting |
| `optical_power_w` | `float` | Watts (1 mW nominal) | Ingestion Stream (ZMQ / MQTT) | Ingested optical input signal power |
| `dark_current_a` | `float` | Amperes | Physics Engine (`evaluate_state`) | Derived thermal leakage current ($I_d = I_{d0} \cdot e^{0.07(T - T_0)}$) |
| `responsivity_a_w`| `float` | A/W | Physics Engine (`evaluate_state`) | Dynamic optical-to-electrical conversion efficiency ($R$) |
| `photocurrent_a` | `float` | Amperes | Physics Engine (`evaluate_state`) | Active signal current generated by incident light ($I_{ph} = P_{opt} \cdot R$) |
| `snr_db` | `float` | dB | Physics Engine (`evaluate_state`) | Real-time Signal-to-Noise Ratio calculated against shot noise |
| `health_index_pct`| `float` | Percentage (0 - 100%) | Physics Engine (`evaluate_state`) | Asset health score inversely proportional to dark current drift |
| `is_saturated` | `boolean` | `True` / `False` | Physics Engine (`evaluate_state`) | Safety condition indicator ($I_{total} \ge 10\text{ mA}$ limit) |
| `recommended_gain_mode`| `string` | `low_noise` \| `balanced` \| `high_sensitivity` | Behavioral AI Planner | Autonomous gain operational recommendation |
| `assigned_bias_v` | `float` | Volts | AI Command Planner | Feedback-controlled bias setting dispatched via MQTT to hardware |

*System Limitation Note:* The assigned_bias_v control is currently operating as an open-loop system. While intended for feedback control to adjust hardware behavior, the telemetry generator presently hardcodes the baseline reading to 5.0 V. The digital twin's commanded bias does not currently alter the physical sensor stream.

--- 
## 2. Shared Physics Engine Verification

To eliminate model drift between this evaluation environment and the microservices, we import PhotodetectorPhysicsEngine directly from services.twin_engine.physics_model.

In [1]:
import sys
import os
import json

# Here we add a service path to import the shared physics model
sys.path.append(os.path.abspath("services/twin_engine"))

from physics_model import PhotodetectorPhysicsEngine

# Next we illustrate the shared physics engine
physics_engine = PhotodetectorPhysicsEngine()
sample_state = physics_engine.evaluate_state(temp_c=25.0, bias_v=5.0, optical_power_w=0.001)

print("Physics Engine Imported Successfully!")
print("Nominal Baseline State Evaluation:")
print(json.dumps(sample_state, indent=2))

Physics Engine Imported Successfully!
Nominal Baseline State Evaluation:
{
  "dark_current_a": 1e-09,
  "responsivity_a_w": 0.91375,
  "photocurrent_a": 0.00091375,
  "total_current_a": 0.000913751,
  "snr_db": 94.53653515622555,
  "is_saturated": false,
  "health_index_pct": 98.5
}


--- 
## 3. Multi-Stream Ingestion, Aggregation, and State Conversion Proof

This section demonstrates real-time streaming acquisition across two transport channels:
1. *ZeroMQ Subscriber (`tcp://localhost:5555`)*: High-throughput transient telemetry.
2. *MQTT Subscriber (`localhost:1883`)*: Decoupled broker messaging on topic telemetry/pd_sensor_01.

Packets from both streams are aggregated to prove ingestion speed, confirm dual-channel identical payloads, and explicitly map the raw inputs into the synthesized digital twin state.

In [2]:
import zmq
import json
import time
import pandas as pd
import paho.mqtt.client as mqtt

# First step is to START THE MQTT SUBSCRIBER IN BACKGROUND ---
mqtt_records = []

def on_mqtt_message(client, userdata, msg):
    # Here we set cap to 40 to match ZMQ sample size
    if len(mqtt_records) >= 40:
        return 
    try:
        payload = json.loads(msg.payload.decode('utf-8'))
        if isinstance(payload, dict):
            payload["stream_source"] = "MQTT (Port 1883)"
            mqtt_records.append(payload)
    except Exception:
        pass

mqtt_client = mqtt.Client(callback_api_version=mqtt.CallbackAPIVersion.VERSION2, client_id="NotebookAggregator")
mqtt_client.on_message = on_mqtt_message

try:
    mqtt_client.connect("localhost", 1883, 60)
    mqtt_client.loop_start() 
    mqtt_client.subscribe("telemetry/pd_sensor_01")
    print("[MQTT] Subscribed to 'telemetry/pd_sensor_01' on localhost:1883. Collecting messages...")
except Exception as e:
    print(f"[MQTT Connection Notice] {e}")

# Second step is to provide the ZEROMQ TELEMETRY STREAM ACQUISITION ---
zmq_context = zmq.Context()
zmq_socket = zmq_context.socket(zmq.SUB)
zmq_socket.setsockopt(zmq.RCVTIMEO, 2000)
zmq_socket.connect("tcp://localhost:5555")
zmq_socket.setsockopt_string(zmq.SUBSCRIBE, "telemetry")

zmq_records = []
print("[ZMQ] Collecting streaming telemetry frames from tcp://localhost:5555...")

for _ in range(40):
    try:
        raw_msg = zmq_socket.recv_string()
        if " " in raw_msg:
            _, payload_str = raw_msg.split(" ", 1)
        else:
            payload_str = raw_msg

        payload = json.loads(payload_str)
        if isinstance(payload, dict):
            payload["stream_source"] = "ZeroMQ (Port 5555)"
            zmq_records.append(payload)
    except Exception:
        pass

zmq_socket.close()
zmq_context.term()
print(f"[ZMQ] Captured {len(zmq_records)} record(s).")

print("[MQTT] Waiting 1 second to finish gathering background MQTT packets...")
time.sleep(1.0)

try:
    mqtt_client.loop_stop()
    mqtt_client.disconnect()
except Exception:
    pass
print(f"[MQTT] Captured {len(mqtt_records)} record(s).")

# Third step is to look at the MULTI-STREAM AGGREGATION & RATE VERIFICATION ---
combined_stream = zmq_records + mqtt_records
df_stream = pd.DataFrame(combined_stream)

if not df_stream.empty:
    # Here we convert timestamps to explicit datetimes to verify the packet temporal spacing
    df_stream["timestamp"] = pd.to_datetime(df_stream["timestamp"], unit="s")

    print(f"\n================ Ingested Multi-Stream Summary ================")
    print(f"Total Packets Acquired Across Channels: {len(df_stream)}")
    
    expected_cols = ["stream_source", "device_id", "temperature_c", "optical_power_w", "timestamp"]
    present_cols = [c for c in expected_cols if c in df_stream.columns]
    
    # we Limit the displayed rows to 20 so it doesn't flood our PDF output, while keeping the data size 80
    pd.set_option('display.max_rows', 20)
    print(df_stream[present_cols])

    # Then we validate dual-transport overlap (Rigorous Check)
    duplicates = df_stream.groupby(["timestamp", "temperature_c", "optical_power_w"])["stream_source"].nunique()
    overlap_count = (duplicates > 1).sum()
    
    if overlap_count > 0:
        print(f"\n[Verification] {overlap_count} identical payloads detected across both transports (matching time and values). This confirms a single true physical signal is being broadcast seamlessly via dual streams.")
    else:
        print("\n[Verification Notice] Both streams captured successfully. (No direct payload overlap in this transient micro-window due to background vs live-blocking collection timing offsets).")

    # Next we calculate the Sampling Rate (Proving the Transient Tier)
    zmq_df = df_stream[df_stream["stream_source"] == "ZeroMQ (Port 5555)"]
    if len(zmq_df) > 1:
        avg_diff_ms = zmq_df["timestamp"].diff().mean().total_seconds() * 1000
        print(f"\n[Performance] Measured ZMQ Sampling Interval: {avg_diff_ms:.2f} ms (~{1000/avg_diff_ms:.1f} Hz)")

    # Fourth step is to look at the EXPLICIT SINGLE PACKET STATE CONVERSIONS ---
    print("\n================ Single Packet State Conversion Proof ================")
    print("Demonstrating the deterministic transformation of one raw sensor packet into the complete Digital Twin physical state.\n")
    
    raw_packet = df_stream.iloc[0]
    single_state = physics_engine.evaluate_state(
        temp_c=raw_packet["temperature_c"],
        bias_v=raw_packet.get("bias_voltage_v", 5.0),
        optical_power_w=raw_packet["optical_power_w"]
    )
    
    print("RAW INGESTED SENSOR PACKET:")
    print(f"  -> Input Temperature: {raw_packet['temperature_c']} °C")
    print(f"  -> Input Power:       {raw_packet['optical_power_w']} W")
    
    print("\nSYNTHESIZED TWIN STATE:")
    for k, v in single_state.items():
        if isinstance(v, float) and "current" in k:
            print(f"  -> {k}: {v:.4e} A")
        elif isinstance(v, float):
            print(f"  -> {k}: {v:.4f}")
        else:
            print(f"  -> {k}: {v}")
else:
    print("No stream packets captured. Ensure 'docker compose up' is active.")

[MQTT] Subscribed to 'telemetry/pd_sensor_01' on localhost:1883. Collecting messages...
[ZMQ] Collecting streaming telemetry frames from tcp://localhost:5555...
[ZMQ] Captured 40 record(s).
[MQTT] Waiting 1 second to finish gathering background MQTT packets...
[MQTT] Captured 40 record(s).

================ Ingested Multi-Stream Summary ================
Total Packets Acquired Across Channels: 80
         stream_source     device_id  temperature_c  optical_power_w  \
0   ZeroMQ (Port 5555)  pd_sensor_01          25.15         0.001081   
1   ZeroMQ (Port 5555)  pd_sensor_01          24.59         0.000998   
2   ZeroMQ (Port 5555)  pd_sensor_01          24.51         0.000952   
3   ZeroMQ (Port 5555)  pd_sensor_01          24.73         0.000981   
4   ZeroMQ (Port 5555)  pd_sensor_01          24.60         0.000995   
..                 ...           ...            ...              ...   
75    MQTT (Port 1883)  pd_sensor_01          24.99         0.001095   
76    MQTT (Port 1883)  p

--- 
## 4. InfluxDB Time-Series Aggregation & Verification

Queries stored telemetry from the `influxdb_service` bucket `photodetector_telemetry`. We utilize a Flux query employing `aggregateWindow` to downsample the high-frequency stream into stable 1-second means, providing a statistical summary describing the volume and variance of the aggregated data.

In [3]:
from influxdb_client import InfluxDBClient

INFLUX_URL = "http://localhost:8086"
INFLUX_TOKEN = "my-super-secret-auth-token"
INFLUX_ORG = "opto-twin"
INFLUX_BUCKET = "photodetector_telemetry"

try:
    client = InfluxDBClient(url=INFLUX_URL, token=INFLUX_TOKEN, org=INFLUX_ORG)
    query_api = client.query_api()

    # Flux Query using aggregateWindow to compute 1-second means over the past 5 minutes
    flux_query = f'''
    from(bucket: "{INFLUX_BUCKET}")
      |> range(start: -5m)
      |> filter(fn: (r) => r["_measurement"] == "photodetector_state")
      |> aggregateWindow(every: 1s, fn: mean, createEmpty: false)
      |> pivot(rowKey:["_time"], columnKey: ["_field"], valueColumn: "_value")
    '''

    # Directly extract to Pandas DataFrame for high-level aggregation metrics
    df_influx = query_api.query_data_frame(flux_query)
    client.close()

    if not df_influx.empty:
        print(f"[InfluxDB Query Success] Connected to {INFLUX_URL}")
        print(f"[Aggregation] Successfully computed {len(df_influx)} downsampled 1-second time windows.\n")
        
        # Clean up database system columns for presentation
        sys_cols = ["result", "table", "_start", "_stop", "_measurement"]
        df_clean = df_influx.drop(columns=[c for c in sys_cols if c in df_influx.columns])
        
        # Output the Describe() Table for Data Volume and Statistics
        print("================ Time-Series Aggregation Statistics ================")
        print(df_clean.describe().round(4))
    else:
        print("[InfluxDB] Query returned no data. Check if twin_engine is actively writing.")
        
except Exception as e:
    print(f"[InfluxDB Query Notice] Could not reach active DB on localhost:8086 ({e})")

[InfluxDB Query Success] Connected to http://localhost:8086
[Aggregation] Successfully computed 301 downsampled 1-second time windows.

================ Time-Series Aggregation Statistics ================
       bias_voltage_v  dark_current_a  health_index_pct  photocurrent_a  \
count           301.0           301.0          301.0000        301.0000   
mean              5.0             0.0           98.5000          0.0009   
std               0.0             0.0            0.0060          0.0000   
min               5.0             0.0           98.4825          0.0009   
25%               5.0             0.0           98.4959          0.0009   
50%               5.0             0.0           98.4996          0.0009   
75%               5.0             0.0           98.5046          0.0009   
max               5.0             0.0           98.5152          0.0009   

       responsivity_a_w    snr_db  temperature_c  
count          301.0000  301.0000       301.0000  
mean             

--- 
## Summary

1. **State Definition**: We defined digital twin schemes by combining raw sensor variables with certain optoelectronic physical equations ($I_d$, $R$, $I_{ph}$, $\text{SNR}$, Health Index) etc.
2. **Zero Drift Physics**: We did unified state calculations by importing it directly from `services.twin_engine.physics_model`.
3. **Multi-Stream Ingestion & Proof**: We also concurrently ingested streams over ZeroMQ and MQTT, while successfully verifying high-speed throughput and detailing exactly how single raw measurements map into a full synthesized twin state.
4. **Data Aggregation Verification**: Lastly, we executed time-windowed downsampling (`aggregateWindow`) on the `influxdb_service`, extracting statistical summaries and verifying accurate time-series telemetry records.